## Quality grading results

In [ ]:
%load_ext autoreload
%autoreload 2


import os
from pathlib import Path

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.options.display.float_format = "{:,.2f}".format

In [ ]:
# Where results of the quality filter are stored
project_dir = Path.cwd()
image_dir = 'datasets/ukb/images'
quality_dir = project_dir.joinpath('quality_results', 'ukb')
n_participants = 1
n_images = 192349

# Threshold for the quality prediction: values below this threshold are considered as bad quality
threshold = 0.25

quality_filters = ['FIT', 'RetiRatey', 'AutoMorph', 'VascX']

### Comparison between quality filters
* Fundus Image Toolbox (BerensLab)
* RetiRatey (UCL)
* AutoMorph
* VascX

#### 1. Preprocess results

In [ ]:
# Clean FIT results
quality_file = quality_dir.joinpath(f'quality_fit.csv')
quality_df = pd.read_csv(quality_file, dtype={'ID': str})

quality_df['ID'] = quality_df['ID'].map(lambda x: x.split(sep='.')[0])

quality_df.to_csv(quality_file, mode='w', index=False, header=True, float_format='%.3f')

In [ ]:
# Set label for RetiRatey
quality_file = quality_dir.joinpath(f'quality_retiratey.csv')
quality_df = pd.read_csv(quality_file, dtype={'ID': str})

quality_df['label'] = ((quality_df['disc'] <= 3) & (quality_df['macula'] <= 3) & (quality_df['vessels'] <= 3)).astype(int)
quality_df['ID'] = quality_df['ID'].map(lambda x: x.split(sep='.')[0])

quality_df.to_csv(quality_file, mode='w', index=False, header=True, float_format='%.3f')

In [ ]:
# Clean automorph results
quality_file = quality_dir.joinpath('quality_automorph.csv')
quality_df = pd.read_csv(quality_file, dtype={'ID': str})

if 'Name' in quality_df.columns:
    quality_df['Name'] = quality_df['Name'].apply(lambda x: os.path.basename(x).split(sep='.')[0])
    quality_df.rename(columns={'Name': 'ID'}, inplace=True)
quality_df['prediction_label'] = quality_df['Prediction'].map({0: 'Good', 1: 'Usable', 2: 'Reject'})
quality_df['label'] = (quality_df['Prediction'] != 2).astype(int)
quality_df['ID'] = quality_df['ID'].map(lambda x: x.split(sep='.')[0])

quality_df.to_csv(quality_file, mode='w', index=False, header=True)

In [ ]:
# Clean VascX results
quality_file = quality_dir.joinpath('quality_vascx.csv')
quality_df = pd.read_csv(quality_file, dtype={'ID': str})

if 'Unnamed: 0' in quality_df.columns:
    quality_df.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
quality_df['prediction'] = quality_df[['q1', 'q2', 'q3']].idxmax(axis=1)
quality_df['prediction_label'] = quality_df['prediction'].map({'q1': 'Good', 'q2': 'Usable', 'q3': 'Reject'})
quality_df['label'] = (quality_df['prediction_label'] != 'Reject').astype(int)
quality_df.to_csv(quality_file, mode='w', index=False, header=True)

#### 2. Load results

In [ ]:
# Load results from all quality filters
results = {}
for quality_filter in quality_filters:
    quality_file = quality_dir.joinpath(f'quality_{quality_filter.lower()}.csv')
    results[f'{quality_filter}'] = pd.read_csv(quality_file, dtype={'ID': str})

#### 3. Percentage of participants with gradable images

In [ ]:
df_quality = pd.DataFrame(columns=['Filter', 'All', 'Usable', 'Usable %', 'Gradable', 'Gradable %'])
df_quality = df_quality.astype(dict(zip(df_quality.columns, [str, int, int, int])))

for quality_filter in quality_filters:
    quality_df = results[quality_filter]
    df_quality_append = pd.DataFrame([{'Filter': quality_filter, 'All': n_images, 'Usable': n_images - 21,
                                      'Gradable': quality_df[quality_df['label'] == 1].shape[0]}])
    df_quality = pd.concat([df_quality, df_quality_append], ignore_index=True)

df_quality['Usable %'] = df_quality['Usable'] * 100 / df_quality['All']
df_quality['Gradable %'] = df_quality['Gradable'] * 100 / df_quality['All']

df_quality

#### 4. Score distribution

In [ ]:
scores_retiratey = pd.DataFrame(columns=['ID', 'disc', 'macula', 'vessels'])
scores_retiratey = scores_retiratey.astype(dict(zip(scores_retiratey.columns, [str, float, float, float])))

# List scores per area for kde plot
quality_df = results['RetiRatey']
score = quality_df['disc'].to_list() + quality_df['macula'].to_list() + quality_df['vessels'].to_list()
region = ['disc'] * quality_df.shape[0] + ['macula'] * quality_df.shape[0] + ['vessels'] * quality_df.shape[0]
scores_retiratey = pd.DataFrame({'Score': score, 'Region': region})

In [ ]:
# Distribution plot
fig, ax = plt.subplots(1, 4, figsize=(16, 4))

# Set thresholds
ax[0].axvspan(-0.2, 0.25, color='crimson', alpha=0.1)
ax[1].axvspan(3, 5, color='crimson', alpha=0.1)
ax[2].axvspan(1.5, 2.5, color='crimson', alpha=0.1)
ax[3].axvspan(1.5, 2.5, color='crimson', alpha=0.1)

sns.kdeplot(x='confidence', data=results['FIT'], ax=ax[0])
sns.kdeplot(x='Score', hue='Region', data=scores_retiratey, ax=ax[1])
sns.histplot(x='prediction_label', data=results['AutoMorph'], stat='percent', shrink=0.6, ax=ax[2])
results['VascX']['prediction_label'] = results['VascX']['prediction_label'].astype('category')
results['VascX']['prediction_label'] = results['VascX']['prediction_label'].cat.reorder_categories(['Good', 'Usable', 'Reject'])
sns.histplot(x='prediction_label', data=results['VascX'], stat='percent', shrink=0.6, ax=ax[3])

ax[0].set_xlim([-0.15, 1])
ax[1].set_xlim([0, 4.9])
ax[2].set_xlim([-0.5, 2.5])
ax[3].set_xlim([-0.5, 2.5])

ax[0].set_xlabel('Score (Ungradable → Gradable)')
ax[1].set_xlabel('Score (Good → Bad)')
ax[2].set_xlabel('')
ax[2].set_ylabel('Percentage of images (%)')
ax[0].set_title('Fundus Image Toolbox')
ax[1].set_title('RetiRatey')
ax[2].set_title('AutoMorph')
ax[3].set_xlabel('')
ax[3].set_title('VascX')

plt.tight_layout()
fig.savefig('plots/quality_distribution.svg')

#### 5. Agreement of predictions

In [ ]:
# Overlap of predictions
# Scores for FIT
df_combined = pd.DataFrame(columns=['ID', 'FIT', 'RetiRatey', 'AutoMorph'])
df_combined = df_combined.astype(dict(zip(df_combined.columns, [str, float, float])))

fit_df = results['FIT'].copy()
fit_df.rename(columns={'label': 'FIT'}, inplace=True)

retiratey_df = results['RetiRatey'].copy()
retiratey_df.rename(columns={'label': 'RetiRatey'}, inplace=True)

automorph_df = results['AutoMorph']
automorph_df.rename(columns={'label': 'AutoMorph'}, inplace=True)

vascx_df = results['VascX']
vascx_df.rename(columns={'label': 'VascX'}, inplace=True)

combined_df = pd.merge(fit_df[['ID', 'FIT']], retiratey_df[['ID', 'RetiRatey']], on="ID")
combined_df = pd.merge(combined_df, automorph_df[['ID', 'AutoMorph']], on="ID")
combined_df = pd.merge(combined_df, vascx_df[['ID', 'VascX']], on="ID")
df_combined = pd.concat([df_combined, combined_df], ignore_index=True) 

df_combined = df_combined.astype({'FIT': bool, 'RetiRatey': bool, 'AutoMorph': bool})

df_overlap = pd.DataFrame(columns=['FIT-RetiRatey', 'FIT-AutoMorph', 'FIT-VascX', 'RetiRatey-AutoMorph', 'RetiRatey-VascX', 'AutoMorph-VascX', 'All'])
df_overlap = df_overlap.astype(dict(zip(df_overlap.columns, [float, float, float, float, float, float, float])))

df_overlap_append = pd.DataFrame({'FIT-RetiRatey': [(df_combined['FIT'] & df_combined['RetiRatey']).sum() *100 / n_images],
                                  'FIT-AutoMorph': [(df_combined['FIT'] & df_combined['AutoMorph']).sum() *100 / n_images],
                                  'FIT-VascX': [(df_combined['FIT'] & df_combined['VascX']).sum() *100 / n_images],
                                  'RetiRatey-AutoMorph': [(df_combined['RetiRatey'] & df_combined['AutoMorph']).sum() *100 / n_images],
                                  'RetiRatey-VascX': [(df_combined['RetiRatey'] & df_combined['VascX']).sum() *100 / n_images],
                                  'AutoMorph-VascX': [(df_combined['AutoMorph'] & df_combined['VascX']).sum() *100 / n_images],
                                  'All': [(df_combined['FIT'] & df_combined['RetiRatey'] & df_combined['AutoMorph'] & df_combined['VascX']).sum() *100 / n_images]})

df_overlap = pd.concat([df_overlap, df_overlap_append], ignore_index=True)

df_overlap

In [ ]:
quality_filters = ['FIT', 'RetiRatey', 'AutoMorph', 'VascX']
cf_matrix = 100 * np.ones((4, 4))

for i, qf1 in enumerate(quality_filters):
    for j, qf2 in enumerate(quality_filters):
        if qf1 != qf2:
            key = f'{qf1}-{qf2}' if f'{qf1}-{qf2}' in df_overlap.columns else f'{qf2}-{qf1}'
            cf_matrix[i, j] = df_overlap.at[df_overlap.index[-1], key]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))

sns.heatmap(cf_matrix, xticklabels=quality_filters, yticklabels=quality_filters, annot=True, fmt='.1f', cmap='rocket_r', annot_kws={"size": 12}, ax=ax)

ax.tick_params(axis='y', rotation=0, labelsize=12)
ax.tick_params(axis='x', labelsize=12)
# ax.set_title('Agreement between quality filters (%)')
plt.tight_layout()
fig.savefig('plots/quality_agreement.svg')